In [ ]:
%use serialization

import java.io.File

private typealias Pattern = String

@Serializable
data class PatternTree(
    @SerialName("w") val word: String,
    @SerialName("g") val nextGuesses: Map<Pattern, PatternTree?>?
)

fun evaluate(guess: String, answer: String): Pattern {
    val correctLetters = guess
        .zip(answer)
        .map { (first, second) -> first == second }

    val remainingLetters = answer.toList()
        .zip(correctLetters)
        .filterNot { (_, correct) -> correct }
        .map { (letter, _) -> letter }

    val (_, result) = guess.toList()
        .zip(correctLetters)
        .fold(remainingLetters to "") { (remainingLetters, result), (letter, correct) ->
            when {
                correct -> remainingLetters to result + 'c'
                letter in remainingLetters -> remainingLetters - letter to result + 'p'
                else -> remainingLetters to result + 'a'
            }
        }

    return result
}

fun calculateEntropy(guess: String, possibleAnswers: List<String>): Double {
    return possibleAnswers
        .groupBy { answer -> evaluate(guess, answer) }
        .values
        .fold(0.0) { sum, possibilities ->
            val probability = possibilities.size.toDouble() / possibleAnswers.size.toDouble()
            sum - probability * log2(probability)
        }
}

fun computePatternTree(currentWord: String, path: List<String>, remainingCandidates: List<String>, wordlist: List<String>, depth: Int = 1): PatternTree? {
    if (depth >= 6) {
        return null
    }

    println(path.joinToString(" ") + " -> " + currentWord)

    val guesses = wordlist
        .minus(currentWord)
        .minus(path)
        .groupBy { evaluate(currentWord, it) }
        .mapNotNull { (pattern, candidates) ->
            if (remainingCandidates.size <= 1) null
            else {
                val nextBestWord = wordlist.maxBy { calculateEntropy(it, candidates) }
                return@mapNotNull pattern to computePatternTree(nextBestWord, path + currentWord, candidates, wordlist, depth + 1)
            }
        }
        .toMap()

    return PatternTree(
        word = currentWord,
        nextGuesses = if (guesses.isEmpty()) null else guesses
    )
}

val json = Json {
    explicitNulls = false
}

val wordlist = json.decodeFromString<List<String>>(File("../../../data/wordle-list.json").readText())
    .map { it.trim() }
    .filter { it.isNotBlank() }

val startingWord = "tares" // wordlist.maxBy { calculateEntropy(it, wordlist) }
val startingPath = emptyList<String>()
val startingCandidates = wordlist
val tree = computePatternTree(startingWord, startingPath, startingCandidates, wordlist)

File("/tmp/wordle-lookup-table.json").writeText(
    json.encodeToString(tree)
)
